In [2]:
import os
import importlib.util
import pathlib
import sys
import socket
import numpy as np
import IMProToo
import xarray as xr
from glob import glob
import gc
import shutil
import psutil


hostname = socket.gethostname()
print(hostname)


# abs_path = str(pathlib.Path(hostname).parent.absolute())
abs_path = str(pathlib.Path(hostname).parent.absolute())

WORKDIR = abs_path[:- (len(abs_path.split('/')[-2] + abs_path.split('/')[-1])+1)]
DATA_DIR = "/mn/vann/franzihe/"


INPUT_DATA_DIR = os.path.join(DATA_DIR, 'input')
OUTPUT_DATA_DIR = os.path.join(DATA_DIR, 'output/4_minutes/')
UTILS_DIR = os.path.join(WORKDIR, 'utils')

sys.path.append(UTILS_DIR)


script_path = pathlib.Path("07_ensemble_approach_get_all_files.py")

spec = importlib.util.spec_from_file_location(
    "ensemble_approach", script_path
)

ensemble = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ensemble)

from imports import pd, time, plt
import functions as fct

ivan.uio.no
ivan.uio.no


In [3]:
# reload imports
%load_ext autoreload
%autoreload 2

In [4]:
site = "ANX"
year = "2023"
version = IMProToo.__version__
base_height = np.arange(100, 3200, 100)  # Predefined MRR profile

In [5]:
a = 1
b = 1

In [6]:
# -------------------------
# Create directories
# -------------------------

nc_dir = pathlib.Path(os.path.join(OUTPUT_DATA_DIR, f'ANX_obs_EM_{a}_{b}'))
nc_dir.mkdir(parents=True, exist_ok=True)

# -------------------------
# 1. Locate files
# -------------------------
file_pattern = f"{INPUT_DATA_DIR}/MRR_ANX/maahn/mrr_improtoo_{version}_{site}_*{year}*.nc"
mrr_files = sorted(glob(file_pattern))

# print(f"📁 Found {len(mrr_files)} raw MRR files")
# assert len(mrr_files) > 0, "No input MRR files found"

mrr_files, met_files, sf_files = ensemble.list_input_files(INPUT_DATA_DIR, site, year, a, b)

mrr_map = ensemble.map_mrr_files_by_date(mrr_files)


# ------------------------------------------------------------------
# 2. Build season-wide event tables (STREAMING)
# ------------------------------------------------------------------
dsx, df_sum, df_mean = ensemble.build_season_tables(mrr_map, met_files,  a, b)
gc.collect()

# ------------------------------------------------------------------
# 3. Load snowflake class database (ONCE)
# ------------------------------------------------------------------
with xr.open_dataset(sf_files[0],
                     engine="h5netcdf", cache=False, decode_timedelta=True) as sf_class:
        sf_class = sf_class[["snowflake_class_name", "fallspeed", "flake_Dmax", "gan3d_mass"]].to_dataframe().sort_index()

# Limit fallspeed
df_mean.loc[df_mean['W_400m'] > 7.0, 'W_400m'] = np.nan
df_mean.loc[df_mean['W_400m'] <= -7.0, 'W_400m'] = np.nan



sf_class.loc[sf_class['fallspeed'] > 7., 'fallspeed'] = np.nan
sf_class.loc[sf_class['fallspeed'] <= 0.2, 'fallspeed'] = np.nan
sf_class = sf_class[sf_class['fallspeed'].notna()]

In [6]:
threshold = 60
mode = "fixed"


In [7]:
for duration in ["4min", ]:#"10min"]:
    df_corr, psd_test = fct.process_precip_segments(
        df_sum,
        df_mean,
        sf_class,
        mode=mode,
        duration=duration,
        a=a,
        b=b,
        threshold=threshold
    )

    expected_h = pd.Timedelta(duration).total_seconds() / 3600

    print(duration)
    print("Windows:", len(df_corr))
    print("Unique durations:", df_corr["delta_h"].unique())
    print("Correct duration:",
          np.isclose(df_corr["delta_h"], expected_h).all())
    print("Unique timestamps:", df_corr.index.is_unique)
    print("PSD aligned:", df_corr.index.equals(psd_test.indexes["time"]))

delta_seconds: 240.0
type: <class 'float'>
4min
Windows: 11160
Unique durations: [0.06666667]
Correct duration: True
Unique timestamps: True
PSD aligned: True


In [8]:
print("Total windows:", len(df_corr))

print(
    "Windows with precipitation >= 0.1 mm/h:",
    df_corr["valid_precip"].sum()
)

print(
    "Windows with MASC classification:",
    df_corr["snowflake_class_name"].notna().sum()
)

print(
    "Windows with valid MASC fallspeed:",
    df_corr["fallspeed"].notna().sum()
)

Total windows: 11160
Windows with precipitation >= 0.1 mm/h: 678
Windows with MASC classification: 1156
Windows with valid MASC fallspeed: 1156


In [9]:
df_sum_offset = df_sum[
        [
            'Matrosov_aggregate_400m', 
            'Bracci_aggregate_400m', 
            'Nakai_aggregate_400m', 
            'Nakai_graupel_400m', 
            'Pu_graupel_400m', 
            'Schoger_400m', 
            f'S_{a}_{b}_400m'
            ]
            ].copy()
df_mean_offset = df_mean[['W_400m']].copy()

In [10]:
df_off = ensemble.apply_offset(
            df_corr, 
            df_sum_offset, 
            df_mean_offset, 
            a, 
            b, 
            mode=mode, 
            duration=duration, )

In [11]:
print(
    df_off[
        [
            "snowflake_class_name",
            "W_class_mean",
            "W_400m",
            "rate_mm_h"
        ]
    ].describe(include="all")
)

       snowflake_class_name  W_class_mean      W_400m   rate_mm_h
count                   402    402.000000  388.000000  402.000000
unique                    4           NaN         NaN         NaN
top                 graupel           NaN         NaN         NaN
freq                    255           NaN         NaN         NaN
mean                    NaN      1.688816    1.803796    2.963811
std                     NaN      0.288216    1.716218    1.711198
min                     NaN      0.956418   -2.529483    0.548500
25%                     NaN      1.442797    0.575496    1.852875
50%                     NaN      1.900585    1.807197    1.963500
75%                     NaN      1.900585    2.989072    3.801000
max                     NaN      1.900585    6.494514    9.465000


In [12]:
pd.crosstab(
    df_off["W_400m"].notna(),
    df_off[f"S_{a}_{b}_400m"].notna(),
    rownames=["Valid shifted W"],
    colnames=["Valid ensemble snowfall rate"]
)

Valid ensemble snowfall rate,False,True
Valid shifted W,,
False,14,0
True,0,388


In [13]:
print(
    df_off.groupby("snowflake_class_name")
    .agg(
        n=("W_class_mean", "size"),
        W_class_mean=("W_class_mean", "mean"),
        W_min=("W_class_mean", "min"),
        W_max=("W_class_mean", "max"),
        rate_mean=("rate_mm_h", "mean")
    )
)

                        n  W_class_mean     W_min     W_max  rate_mean
snowflake_class_name                                                  
aggregate              61      1.442797  1.442797  1.442797   2.598311
graupel               255      1.900585  1.900585  1.900585   3.176843
mixed                  80      1.256319  1.256319  1.256319   2.625475
pristine                6      0.956418  0.956418  0.956418   2.137000
